# GSB 5544 — Topic 3.1 (Extended): Joining and Merging Data, step by step
*Every operation with the question that motivates it, the tables before, and the table after — in colour.*

## What this extended version adds

The standard Topic 3.1 notebook shows each technique once. This version slows down and, for **every** operation, answers four questions *before* running anything:

1. **What question are we trying to answer?** — the reason a single table is not enough.
2. **Which tables hold the pieces, and what is the key** that links them?
3. **What do we expect** the result to look like (how many rows, which columns, where the `NaN`s will be)?
4. **What actually happened** — read off the colour-coded result and check it against the expectation.

Each operation shows the input table(s) first, then the result, with columns shaded by the table they came from. Watch the same rows move on the interactive page: [https://gato365.github.io/gsb5544_instructor_learn_prep/week_3/pandas.html](https://gato365.github.io/gsb5544_instructor_learn_prep/week_3/pandas.html) (every section links to its technique). The standard notebook and its student version are on the course site: [https://gato365.github.io/gsb5544_instructor_learn_prep/](https://gato365.github.io/gsb5544_instructor_learn_prep/).

### How to read the tables in this notebook

Every table is drawn with **ruled borders** and a **coloured header** that says which DataFrame it is. When a table is the *result* of an operation, each **column is shaded by the table it came from**, so you can see at a glance which pieces of the two inputs were glued together and which columns the operation created.

| Colour | Meaning |
|---|---|
| 🟦 blue | columns from the **left** (first) table |
| 🟧 orange | columns from the **right** (second) table |
| 🟩 green | the **key** column(s) the operation matched on (thick borders too) |
| 🟨 yellow | columns **created or renamed** by the operation (e.g. `_merge`, `score_mid`, a new count) |
| 🟪 purple header | a **result** table |
| 🟥 pink cell | a **missing value** (`NaN`) — usually a row that found no partner |

The caption of every table shows the full row × column count even when only the first rows are printed. Run the next cell once; it defines the `show()` helper used throughout.

In [1]:
import pandas as pd
import numpy as np
from IPython.display import display, HTML

COLORS = {"blue": "#dbe9ff", "orange": "#ffe0c2", "green": "#d9f2d9",
          "purple": "#ead9f7", "yellow": "#fff3b0", "grey": "#eeeeee"}

def show(df, title, header="blue", sources=None, key=None, rows=8):
    """Display `df` as a captioned, ruled, colour-coded table.

    header  : colour name for the header row (which table this is)
    sources : {column: colour} — shades each column of a RESULT by where it came from
    key     : key column(s) — drawn bold with thick borders
    rows    : how many rows to print (the caption always shows the true size)
    """
    if isinstance(df, pd.Series):
        df = df.to_frame()
    shown = df.head(rows)
    hdr = COLORS[header]
    styler = (shown.style
              .set_caption(f"{title} — {df.shape[0]:,} rows × {df.shape[1]} columns"
                           + ("" if len(df) <= rows else f" (first {rows} shown)"))
              .set_table_styles([
                  {"selector": "caption", "props": [("caption-side", "top"), ("text-align", "left"),
                                                    ("font-weight", "bold"), ("font-size", "1.05em"),
                                                    ("padding", "6px 0"), ("color", "#222")]},
                  {"selector": "th", "props": [("background-color", hdr), ("border", "1px solid #777"),
                                               ("text-align", "center"), ("padding", "4px 8px")]},
                  {"selector": "td", "props": [("border", "1px solid #bbb"), ("padding", "3px 8px")]},
                  {"selector": "", "props": [("border", "2px solid #444"), ("border-collapse", "collapse"),
                                             ("margin-bottom", "14px")]},
              ]))
    if sources:
        styler = styler.apply(lambda col: [f"background-color: {COLORS[sources.get(col.name, 'yellow')]}"] * len(col),
                              axis=0)
    if key:
        keys = [key] if isinstance(key, str) else list(key)
        keys = [k for k in keys if k in shown.columns]
        if keys:
            styler = styler.set_properties(subset=keys, **{"font-weight": "bold",
                                                            "border-left": "2px solid #444",
                                                            "border-right": "2px solid #444"})
    styler = styler.highlight_null(color="#ffcdd2")
    display(styler)

def sources_from(left, right, key, left_color="blue", right_color="orange"):
    """Colour map for the RESULT of combining `left` and `right` on `key`."""
    keys = [key] if isinstance(key, str) else list(key)
    m = {}
    for c in left.columns:
        m[c] = left_color
    for c in right.columns:
        m[c] = right_color
    for k in keys:
        m[k] = "green"
    return m

def legend():
    items = [("blue", "left table"), ("orange", "right table"), ("green", "key column"),
             ("yellow", "created / renamed"), ("purple", "result header")]
    html = "".join(f'<span style="display:inline-block;margin:2px 10px 2px 0">'
                   f'<span style="display:inline-block;width:14px;height:14px;border:1px solid #666;'
                   f'background:{COLORS[c]};vertical-align:middle"></span> {t}</span>' for c, t in items)
    html += ('<span style="display:inline-block;margin:2px 10px 2px 0"><span style="display:inline-block;'
             'width:14px;height:14px;border:1px solid #666;background:#ffcdd2;vertical-align:middle"></span> missing value</span>')
    display(HTML(html))

legend()
print("show() is ready")

show() is ready


---
## 1. The situation: two tables, one question

A department keeps a **roster** of students and, separately, a **score sheet**. Neither table alone can answer *"what did each named student score?"* — the roster has names but no scores, the score sheet has scores but no names. They are linked only by `student_id`. That link is the **key**, and using it to combine tables is a **join**.

In [2]:
students = pd.DataFrame({"student_id": [1, 2, 3], "name": ["Ava", "Ben", "Cam"]})
grades   = pd.DataFrame({"student_id": [2, 3, 4], "score": [80, 90, 70]})

show(students, "students  (LEFT table: the roster)", "blue", key="student_id")
show(grades,   "grades  (RIGHT table: the score sheet)", "orange", key="student_id")

,student_id,name
0,1,Ava
1,2,Ben
2,3,Cam


,student_id,score
0,2,80
1,3,90
2,4,70


**Look at the key column before doing anything.** In `students` the IDs are 1, 2, 3; in `grades` they are 2, 3, 4.

- IDs **2 and 3** are in *both* tables — these rows will match.
- ID **1** (Ava) is only in the roster — no score exists.
- ID **4** is only in the score sheet — a score with no roster entry (a typo? a dropped student?).

Every join below is a different policy for rows 1 and 4. Write down what you expect *before* each result: it is the fastest way to learn what a join does.

---
## 2. The four joins — same inputs, four policies for unmatched rows

The inputs are the two tables above; they do not change in this section, so they are not re-printed. The key is `student_id` in both.

### 2a. Inner join  🎬 [https://gato365.github.io/gsb5544_instructor_learn_prep/week_3/pandas.html#inner-join](https://gato365.github.io/gsb5544_instructor_learn_prep/week_3/pandas.html#inner-join)

**Question:** *Among students on the roster, who has a score, and what is it?*
**Why a join:** names are in `students`, scores in `grades`.
**Expect:** only IDs found in both tables → 2 rows (Ben, Cam); columns `student_id`, `name`, `score`; no `NaN`.

In [3]:
inner = students.merge(grades, on="student_id", how="inner")
show(inner, 'students.merge(grades, on="student_id", how="inner")', "purple",
     sources=sources_from(students, grades, "student_id"), key="student_id")

,student_id,name,score
0,2,Ben,80
1,3,Cam,90


**What happened:** pandas walked the roster row by row and looked each `student_id` up in `grades`. Rows 2 and 3 found a partner and were glued together — blue columns from the left, orange from the right, the key (green) appears once. Row 1 (no partner) and row 4 (not in the roster at all) were dropped. `how="inner"` is the default, which is why an unadorned `.merge()` quietly loses rows.

### 2b. Left join  🎬 [https://gato365.github.io/gsb5544_instructor_learn_prep/week_3/pandas.html#left-join](https://gato365.github.io/gsb5544_instructor_learn_prep/week_3/pandas.html#left-join)

**Question:** *Give me the complete roster, with scores where they exist — which students are still missing a score?*
**Why a join:** same two tables, but now the roster is the thing we must not lose.
**Expect:** all 3 roster rows; Ava's `score` is `NaN` (pink); ID 4 still absent because it is not on the roster.

In [4]:
left = students.merge(grades, on="student_id", how="left")
show(left, 'students.merge(grades, on="student_id", how="left")', "purple",
     sources=sources_from(students, grades, "student_id"), key="student_id")
left.dtypes

,student_id,name,score
0,1,Ava,nan
1,2,Ben,80.000000
2,3,Cam,90.000000


student_id      int64
name              str
score         float64
dtype: object

**What happened:** every left row survives. Ava had no partner, so her right-side columns were filled with `NaN`. Notice the side effect in `dtypes`: `score` was an integer column but a column containing `NaN` must be stored as **float** — hence `80.0`. The missing score is the *answer* to the question: it points at the student without a score.

### 2c. Right join  🎬 [https://gato365.github.io/gsb5544_instructor_learn_prep/week_3/pandas.html#right-join](https://gato365.github.io/gsb5544_instructor_learn_prep/week_3/pandas.html#right-join)

**Question:** *Keep every score record — which scores belong to an ID that is not on the roster?*
**Why a join:** now the score sheet is the thing we must not lose (auditing the scores, not the roster).
**Expect:** all 3 score rows; ID 4 has `NaN` for `name`; Ava absent.

In [5]:
right = students.merge(grades, on="student_id", how="right")
show(right, 'students.merge(grades, on="student_id", how="right")', "purple",
     sources=sources_from(students, grades, "student_id"), key="student_id")

,student_id,name,score
0,2,Ben,80
1,3,Cam,90
2,4,nan,70


**What happened:** the mirror image of the left join. The pink `NaN` under `name` is a score recorded for someone who is not a student — exactly the kind of data-quality problem a right join exposes. (`students.merge(grades, how="right")` is the same as `grades.merge(students, how="left")` apart from column order.)

### 2d. Outer (full) join  🎬 [https://gato365.github.io/gsb5544_instructor_learn_prep/week_3/pandas.html#full-outer-join](https://gato365.github.io/gsb5544_instructor_learn_prep/week_3/pandas.html#full-outer-join)

**Question:** *Across both files, which IDs appear only in the roster, only in the scores, and in both?*
**Why a join:** an audit of both sources at once.
**Expect:** 4 rows (IDs 1–4), `NaN` on whichever side is missing, rows sorted by key.

In [6]:
outer = students.merge(grades, on="student_id", how="outer")
show(outer, 'students.merge(grades, on="student_id", how="outer")', "purple",
     sources=sources_from(students, grades, "student_id"), key="student_id")

,student_id,name,score
0,1,Ava,nan
1,2,Ben,80.000000
2,3,Cam,90.000000
3,4,nan,70.000000


**What happened:** nothing was dropped. Each pink cell marks a one-sided row. A useful identity when keys are unique on both sides: `rows(outer) = rows(left) + rows(right) − rows(inner)` = 3 + 3 − 2 = 4.

**Check yourself:** the four results have 2, 3, 3, 4 rows. Say, for each, which of the two "odd" IDs (1 and 4) survived and why.

---
## 3. Anti joins — when the *unmatched* rows are the answer

### 3a. Left anti join  🎬 [https://gato365.github.io/gsb5544_instructor_learn_prep/week_3/pandas.html#left-anti-join](https://gato365.github.io/gsb5544_instructor_learn_prep/week_3/pandas.html#left-anti-join)

**Question:** *Which students have not received a score yet?* (Someone has to chase them.)
**Why not a plain join:** a join returns the matches; we want the leftovers. pandas has no `how="anti"`, so we filter: keep roster rows whose ID is **not in** the score sheet.
**Expect:** just Ava, with only the roster's columns.

In [7]:
mask = students["student_id"].isin(grades["student_id"])
show(pd.DataFrame({"student_id": students["student_id"], "in_grades": mask}),
     'Step 1: students["student_id"].isin(grades["student_id"])', "grey", key="student_id")

left_anti = students[~mask]
show(left_anti, "Step 2: students[~mask]  (rows with NO partner in grades)", "purple",
     sources={"student_id": "green", "name": "blue"}, key="student_id")

,student_id,in_grades
0,1,False
1,2,True
2,3,True


,student_id,name
0,1,Ava


**What happened:** `.isin()` produced one True/False per roster row; `~` flips it; the boolean filter keeps the False rows. Only left-table columns exist because no join took place.

### 3b. Right anti join  🎬 [{PAGE}#right-anti-join]({PAGE}#right-anti-join)

**Question:** *Which scores were recorded under an ID that no student has?* **Expect:** only ID 4.

In [8]:
right_anti = grades[~grades["student_id"].isin(students["student_id"])]
show(right_anti, 'grades[~grades["student_id"].isin(students["student_id"])]', "purple",
     sources={"student_id": "green", "score": "orange"}, key="student_id")

,student_id,score
2,4,70


**Same answer via the outer join:** add `indicator=True` and keep the `left_only` (or `right_only`) rows. This scales to multi-column keys, where `.isin()` on a single column is not enough.

In [9]:
audit = students.merge(grades, on="student_id", how="outer", indicator=True)
show(audit, "outer join with indicator=True", "purple",
     sources=sources_from(students, grades, "student_id"), key="student_id")
show(audit[audit["_merge"] == "left_only"], 'audit[audit["_merge"] == "left_only"]  (= left anti join)', "purple",
     sources=sources_from(students, grades, "student_id"), key="student_id")

,student_id,name,score,_merge
0,1,Ava,nan,left_only
1,2,Ben,80.000000,both
2,3,Cam,90.000000,both
3,4,nan,70.000000,right_only


,student_id,name,score,_merge
0,1,Ava,nan,left_only


---
## 4. Cross join — every combination  🎬 [https://gato365.github.io/gsb5544_instructor_learn_prep/week_3/pandas.html#cross-join](https://gato365.github.io/gsb5544_instructor_learn_prep/week_3/pandas.html#cross-join)

**Question:** *If every student could take every course, what would the full enrollment grid look like?* (Useful for building a template you then fill in, or for pricing every product × every region.)
**Why not a key:** there is nothing to match on — we *want* all pairs.
**Expect:** 3 names × 2 courses = 6 rows.

In [10]:
names   = pd.DataFrame({"name": ["Ava", "Ben", "Cam"]})
courses = pd.DataFrame({"course": ["Stats", "Python"]})
show(names, "names (LEFT)", "blue")
show(courses, "courses (RIGHT)", "orange")

grid = names.merge(courses, how="cross")
show(grid, 'names.merge(courses, how="cross")', "purple", sources={"name": "blue", "course": "orange"})

,name
0,Ava
1,Ben
2,Cam


,course
0,Stats
1,Python


,name,course
0,Ava,Stats
1,Ava,Python
2,Ben,Stats
3,Ben,Python
4,Cam,Stats
5,Cam,Python


**What happened:** each left row was paired with each right row; no key, no `NaN`, and the row count is the *product* of the two sizes — which is why a cross join on two large tables is dangerous.

---
## 5. Details that break real joins

### 5a. The key has a different name in each table  🎬 [https://gato365.github.io/gsb5544_instructor_learn_prep/week_3/pandas.html#different-key-names](https://gato365.github.io/gsb5544_instructor_learn_prep/week_3/pandas.html#different-key-names)

**Question:** *Attach scores to the roster when the score file calls the ID column `id`.*
**Why it matters:** `on="student_id"` would raise `KeyError` because the right table has no such column.
**Expect:** the same left join as 2b, but **both** key columns survive (`student_id` and `id`).

In [11]:
grades_id = pd.DataFrame({"id": [2, 3, 4], "score": [80, 90, 70]})
show(students, "students (LEFT) — key is student_id", "blue", key="student_id")
show(grades_id, "grades_id (RIGHT) — key is id", "orange", key="id")

diff_names = students.merge(grades_id, how="left", left_on="student_id", right_on="id")
show(diff_names, 'students.merge(grades_id, how="left", left_on="student_id", right_on="id")', "purple",
     sources={"student_id": "green", "name": "blue", "id": "green", "score": "orange"}, key=["student_id", "id"])

,student_id,name
0,1,Ava
1,2,Ben
2,3,Cam


,id,score
0,2,80
1,3,90
2,4,70


,student_id,name,id,score
0,1,Ava,nan,nan
1,2,Ben,2.000000,80.000000
2,3,Cam,3.000000,90.000000


**What happened:** pandas matched `student_id` against `id`, but because the names differ it cannot collapse them into one column, so both stay. Ava's `id` is `NaN` because that value came from the right side. Drop one afterwards with `.drop(columns="id")`.

### 5b. The key is a combination of columns  🎬 [https://gato365.github.io/gsb5544_instructor_learn_prep/week_3/pandas.html#multiple-keys](https://gato365.github.io/gsb5544_instructor_learn_prep/week_3/pandas.html#multiple-keys)

**Question:** *What score did each student earn in each course they are enrolled in?*
**Why one key is not enough:** Ava is enrolled in two courses and has two grades. Matching on `student_id` alone cannot tell which grade belongs to which enrollment.
**Expect (correct):** 3 rows, one per enrollment, each with the matching course's score. **Expect (wrong, single key):** Ava's two enrollments × Ava's two grades = 4 Ava rows.

In [12]:
enrollment    = pd.DataFrame({"student_id": [1, 1, 2], "course": ["Stats", "Python", "Stats"]})
course_grades = pd.DataFrame({"student_id": [1, 1, 2], "course": ["Python", "Stats", "Stats"], "score": [95, 80, 90]})
show(enrollment, "enrollment (LEFT)", "blue", key=["student_id", "course"])
show(course_grades, "course_grades (RIGHT)", "orange", key=["student_id", "course"])

two_keys = enrollment.merge(course_grades, how="left", on=["student_id", "course"])
show(two_keys, 'on=["student_id", "course"]  (correct)', "purple",
     sources=sources_from(enrollment, course_grades, ["student_id", "course"]), key=["student_id", "course"])

one_key = enrollment.merge(course_grades, how="left", on="student_id")
show(one_key, 'on="student_id"  (WRONG: courses mixed, rows multiplied)', "purple",
     sources={"student_id": "green", "course_x": "yellow", "course_y": "yellow", "score": "orange"}, key="student_id")

,student_id,course
0,1,Stats
1,1,Python
2,2,Stats


,student_id,course,score
0,1,Python,95
1,1,Stats,80
2,2,Stats,90


,student_id,course,score
0,1,Stats,80
1,1,Python,95
2,2,Stats,90


,student_id,course_x,course_y,score
0,1,Stats,Python,95
1,1,Stats,Stats,80
2,1,Python,Python,95
3,1,Python,Stats,80
4,2,Stats,Stats,90


**What happened:** with both columns in the key, each enrollment found exactly one grade. With one key, every Ava enrollment matched every Ava grade, producing 5 rows, and the two `course` columns collided and were renamed `course_x` / `course_y` (yellow) — a sure sign the key was incomplete.

### 5c. Where did each row come from, and is the key really unique?  🎬 [https://gato365.github.io/gsb5544_instructor_learn_prep/week_3/pandas.html#audit-validate](https://gato365.github.io/gsb5544_instructor_learn_prep/week_3/pandas.html#audit-validate)

**Question:** *After the join, can I prove nothing was lost or duplicated?*
**Why:** joins fail silently. `indicator=True` labels every row's origin; `validate=` asserts the key relationship you expect (`"one_to_one"`, `"one_to_many"`, `"many_to_one"`) and raises `MergeError` if the data disagrees.

In [13]:
audit = students.merge(grades, on="student_id", how="outer", indicator=True, validate="one_to_many")
show(audit, 'outer join, indicator=True, validate="one_to_many"', "purple",
     sources=sources_from(students, grades, "student_id"), key="student_id")
audit["_merge"].value_counts()

,student_id,name,score,_merge
0,1,Ava,nan,left_only
1,2,Ben,80.000000,both
2,3,Cam,90.000000,both
3,4,nan,70.000000,right_only


_merge
both          2
left_only     1
right_only    1
Name: count, dtype: int64

### 5d. Duplicate keys multiply rows  🎬 [https://gato365.github.io/gsb5544_instructor_learn_prep/week_3/pandas.html#duplicate-matches](https://gato365.github.io/gsb5544_instructor_learn_prep/week_3/pandas.html#duplicate-matches)

**Question:** *Why did my row count jump after a join?*
**Setup:** Ben (ID 2) is in two clubs and has two exam scores.
**Expect:** joining on `student_id` gives 2 × 2 = **4** rows for one student — every club paired with every exam.

In [14]:
clubs       = pd.DataFrame({"student_id": [2, 2], "club": ["Art", "Music"]})
exam_grades = pd.DataFrame({"student_id": [2, 2], "exam": ["E1", "E2"], "score": [80, 90]})
show(clubs, "clubs (LEFT) — key repeated!", "blue", key="student_id")
show(exam_grades, "exam_grades (RIGHT) — key repeated!", "orange", key="student_id")

blowup = clubs.merge(exam_grades, on="student_id")
show(blowup, 'clubs.merge(exam_grades, on="student_id")  — 2 × 2 = 4 rows', "purple",
     sources=sources_from(clubs, exam_grades, "student_id"), key="student_id")

try:
    clubs.merge(exam_grades, on="student_id", validate="one_to_many")
except pd.errors.MergeError as e:
    print("validate caught it →", str(e).splitlines()[0])

,student_id,club
0,2,Art
1,2,Music


,student_id,exam,score
0,2,E1,80
1,2,E2,90


,student_id,club,exam,score
0,2,Art,E1,80
1,2,Art,E2,90
2,2,Music,E1,80
3,2,Music,E2,90


validate caught it → Merge keys are not unique in left dataset; not a one-to-many merge


**What happened:** a join produces one output row per *matching pair*. Two left rows × two right rows with the same key = four pairs. Nothing is wrong syntactically, which is why `validate=` exists: it turns a silent multiplication into an error.

### 5e. Both tables have a column with the same name  🎬 [https://gato365.github.io/gsb5544_instructor_learn_prep/week_3/pandas.html#overlapping-columns-suffixes](https://gato365.github.io/gsb5544_instructor_learn_prep/week_3/pandas.html#overlapping-columns-suffixes)

**Question:** *How did each student's score change from the midterm to the final?*
**Why it matters:** both tables call their column `score`; pandas must rename one or both. The default suffixes `_x` / `_y` tell you nothing — choose your own.

In [15]:
midterm = pd.DataFrame({"student_id": [1, 2, 3], "score": [72, 85, 90]})
final   = pd.DataFrame({"student_id": [1, 2, 3], "score": [80, 79, 95]})
show(midterm, "midterm (LEFT)", "blue", key="student_id")
show(final, "final (RIGHT)", "orange", key="student_id")

both = midterm.merge(final, on="student_id", suffixes=("_mid", "_final"))
both["change"] = both["score_final"] - both["score_mid"]
show(both, 'suffixes=("_mid", "_final"), then change = score_final − score_mid', "purple",
     sources={"student_id": "green", "score_mid": "blue", "score_final": "orange", "change": "yellow"}, key="student_id")

,student_id,score
0,1,72
1,2,85
2,3,90


,student_id,score
0,1,80
1,2,79
2,3,95


,student_id,score_mid,score_final,change
0,1,72,80,8
1,2,85,79,-6
2,3,90,95,5


---
## 6. Concatenation — stacking, not matching

### 6a. Add rows  🎬 [https://gato365.github.io/gsb5544_instructor_learn_prep/week_3/pandas.html#concat-add-rows](https://gato365.github.io/gsb5544_instructor_learn_prep/week_3/pandas.html#concat-add-rows)

**Question:** *What is the combined list of scores across fall and spring?*
**Why not a join:** the two tables hold the *same kind* of rows for different periods. There is no key to match; we want them one after another.
**Expect:** 2 + 1 = 3 rows, same columns, a fresh 0-1-2 index.

In [16]:
fall   = pd.DataFrame({"student": ["Ava", "Ben"], "score": [80, 90]})
spring = pd.DataFrame({"student": ["Cam"], "score": [70]})
show(fall, "fall", "blue")
show(spring, "spring", "orange")

stacked = pd.concat([fall, spring], ignore_index=True)
stacked["term"] = ["fall", "fall", "spring"]
show(stacked, "pd.concat([fall, spring], ignore_index=True)  (+ a term column added afterwards)", "purple",
     sources={"student": "purple", "score": "purple", "term": "yellow"})

,student,score
0,Ava,80
1,Ben,90


,student,score
0,Cam,70


,student,score,term
0,Ava,80,fall
1,Ben,90,fall
2,Cam,70,spring


**What happened:** rows were appended; columns aligned by *name*. Without `ignore_index=True` the index would read 0, 1, 0. Good practice: add a column saying which piece each row came from (done here as `term`) *before* or right after stacking, so the origin is not lost.

### 6b. Add columns  🎬 [https://gato365.github.io/gsb5544_instructor_learn_prep/week_3/pandas.html#concat-add-columns](https://gato365.github.io/gsb5544_instructor_learn_prep/week_3/pandas.html#concat-add-columns)

**Question:** *Two measurements share index labels but not row positions — how do we put them side by side?*
**Why it is a trap:** `axis=1` aligns on the **index labels**, not on row order. Set the index to the identifier first.

In [17]:
names_idx  = pd.DataFrame({"index": [0, 1], "name": ["Ava", "Ben"]})
scores_idx = pd.DataFrame({"index": [1, 2], "score": [90, 70]})
show(names_idx, "names_idx", "blue", key="index")
show(scores_idx, "scores_idx", "orange", key="index")

side = pd.concat([names_idx.set_index("index"), scores_idx.set_index("index")], axis=1).reset_index()
show(side, "pd.concat([... set_index('index') ...], axis=1)", "purple",
     sources={"index": "green", "name": "blue", "score": "orange"}, key="index")

,index,name
0,0,Ava
1,1,Ben


,index,score
0,1,90
1,2,70


,index,name,score
0,0,Ava,nan
1,1,Ben,90.000000
2,2,nan,70.000000


---
## 7. Reshaping — same data, different shape

### 7a. Pivot: long → wide  🎬 [https://gato365.github.io/gsb5544_instructor_learn_prep/week_3/pandas.html#pivot-long-wide](https://gato365.github.io/gsb5544_instructor_learn_prep/week_3/pandas.html#pivot-long-wide)

**Question:** *Did each student improve from exam 1 to exam 2?* — easiest to see with E1 and E2 side by side.
**Expect:** one row per student, one column per exam; `pivot` does **not** aggregate, so (student, exam) pairs must be unique.

In [18]:
exam_long = pd.DataFrame({"student": ["Ava", "Ava", "Ben", "Ben"], "exam": ["E1", "E2", "E1", "E2"],
                          "score": [80, 90, 70, 100]})
show(exam_long, "exam_long (one row per student × exam)", "blue", key=["student", "exam"])

wide = exam_long.pivot(index="student", columns="exam", values="score").reset_index()
wide.columns.name = None
show(wide, 'exam_long.pivot(index="student", columns="exam", values="score")', "purple",
     sources={"student": "green", "E1": "yellow", "E2": "yellow"}, key="student")

,student,exam,score
0,Ava,E1,80
1,Ava,E2,90
2,Ben,E1,70
3,Ben,E2,100


,student,E1,E2
0,Ava,80,90
1,Ben,70,100


### 7b. Pivot table: aggregate while reshaping  🎬 [https://gato365.github.io/gsb5544_instructor_learn_prep/week_3/pandas.html#pivot-table-aggregate](https://gato365.github.io/gsb5544_instructor_learn_prep/week_3/pandas.html#pivot-table-aggregate)

**Question:** *Ava retook E1 — what is each student's average per exam?* Duplicated pairs make `pivot` raise; `pivot_table` summarizes them.

In [19]:
exam_dup = pd.DataFrame({"student": ["Ava", "Ava", "Ben", "Ben", "Ava"], "exam": ["E1", "E2", "E1", "E2", "E1"],
                         "score": [80, 90, 70, 100, 100]})
show(exam_dup, "exam_dup — (Ava, E1) appears twice", "blue", key=["student", "exam"])

try:
    exam_dup.pivot(index="student", columns="exam", values="score")
except ValueError as e:
    print("pivot →", e)

pt = exam_dup.pivot_table(index="student", columns="exam", values="score", aggfunc="mean").reset_index()
pt.columns.name = None
show(pt, 'pivot_table(..., aggfunc="mean")  — Ava E1 = mean(80, 100) = 90', "purple",
     sources={"student": "green", "E1": "yellow", "E2": "yellow"}, key="student")

,student,exam,score
0,Ava,E1,80
1,Ava,E2,90
2,Ben,E1,70
3,Ben,E2,100
4,Ava,E1,100


pivot → Index contains duplicate entries, cannot reshape


,student,E1,E2
0,Ava,90.000000,90.000000
1,Ben,70.000000,100.000000


### 7c. Melt: wide → long  🎬 [https://gato365.github.io/gsb5544_instructor_learn_prep/week_3/pandas.html#melt-wide-long](https://gato365.github.io/gsb5544_instructor_learn_prep/week_3/pandas.html#melt-wide-long)

**Question:** *How do we plot every exam score as one series, coloured by exam?* Plotting libraries want *long* data: one row per observation.

In [20]:
show(wide, "wide (input)", "blue", key="student")
long_again = wide.melt(id_vars="student", var_name="exam", value_name="score")
show(long_again, 'wide.melt(id_vars="student", var_name="exam", value_name="score")', "purple",
     sources={"student": "green", "exam": "yellow", "score": "yellow"}, key="student")

,student,E1,E2
0,Ava,80,90
1,Ben,70,100


,student,exam,score
0,Ava,E1,80
1,Ben,E1,70
2,Ava,E2,90
3,Ben,E2,100


### 7d. Drop duplicates — making a key unique before a join  🎬 [https://gato365.github.io/gsb5544_instructor_learn_prep/week_3/pandas.html#drop-duplicates](https://gato365.github.io/gsb5544_instructor_learn_prep/week_3/pandas.html#drop-duplicates)

**Question:** *Is this roster safe to use as a lookup table?* A lookup must have each key **once**, otherwise Section 5d happens.

In [21]:
roster = pd.DataFrame({"student_id": [1, 2, 2, 3, 1], "name": ["Ava", "Ben", "Ben", "Cam", "Ava"]})
show(roster, "roster — IDs 1 and 2 repeated", "blue", key="student_id")
show(roster.drop_duplicates(subset="student_id"), 'roster.drop_duplicates(subset="student_id")  (first occurrence kept)',
     "purple", sources={"student_id": "green", "name": "blue"}, key="student_id")

,student_id,name
0,1,Ava
1,2,Ben
2,2,Ben
3,3,Cam
4,1,Ava


,student_id,name
0,1,Ava
1,2,Ben
3,3,Cam


### 7e. Crosstab — counting pairs of categories  🎬 [https://gato365.github.io/gsb5544_instructor_learn_prep/week_3/pandas.html#crosstab-counts](https://gato365.github.io/gsb5544_instructor_learn_prep/week_3/pandas.html#crosstab-counts)

**Question:** *How many times is each student enrolled in each course?* A crosstab is a pivot table whose cells are counts.

In [22]:
enroll2 = pd.DataFrame({"student": ["Ava", "Ava", "Ben", "Cam", "Cam"],
                        "course": ["Stats", "Python", "Stats", "Python", "Stats"]})
show(enroll2, "enroll2", "blue", key=["student", "course"])
ct = pd.crosstab(enroll2["student"], enroll2["course"]).reset_index()
ct.columns.name = None
show(ct, 'pd.crosstab(enroll2["student"], enroll2["course"])', "purple",
     sources={"student": "green", "Python": "yellow", "Stats": "yellow"}, key="student")

,student,course
0,Ava,Stats
1,Ava,Python
2,Ben,Stats
3,Cam,Python
4,Cam,Stats


,student,Python,Stats
0,Ava,1,1
1,Ben,0,1
2,Cam,1,1


---
## 8. Real data: why we join Kobe Bryant's game log to a team table

**The question:** *Which division did Kobe score the most against, and which did he beat most often?*

**Why one table cannot answer it:** the game log has one row per game with the opponent as a three-letter abbreviation — no division anywhere. The team table has divisions but no games. The abbreviation is the only bridge, so it is the key.

**Expect:** a left join keeps all 1,103 games (we must not lose games); each game gains `team`, `conference`, `division`. Any `NaN` in those columns means an abbreviation the team table does not know.

In [23]:
DATA = "https://raw.githubusercontent.com/gato365/gsb5544_instructor_learn_prep/main/assignments/Data/"
games = pd.read_csv(DATA + "kobe_bryant_games.csv")
teams = pd.read_csv(DATA + "nba_teams.csv")
moves = pd.read_csv(DATA + "franchise_moves.csv")

game_cols = ["season_label", "date", "location", "opponent", "outcome", "points"]
show(games[game_cols], "games (LEFT) — selected columns; key is opponent", "blue", key="opponent")
show(teams, "teams (RIGHT) — key is abbr", "orange", key="abbr")

,season_label,date,location,opponent,outcome,points
0,1996-97,1996-11-03,Home,MIN,W,0
1,1996-97,1996-11-05,Away,NYK,W,1
2,1996-97,1996-11-06,Away,CHH,L,5
3,1996-97,1996-11-08,Away,TOR,L,10
4,1996-97,1996-11-10,Home,ATL,W,2
5,1996-97,1996-11-12,Away,HOU,W,2
6,1996-97,1996-11-13,Away,SAS,L,6
7,1996-97,1996-11-15,Home,LAC,W,4


,abbr,team,city,conference,division
0,ATL,Atlanta Hawks,Atlanta,Eastern,Southeast
1,BOS,Boston Celtics,Boston,Eastern,Atlantic
2,BRK,Brooklyn Nets,Brooklyn,Eastern,Atlantic
3,CHO,Charlotte Hornets,Charlotte,Eastern,Southeast
4,CHI,Chicago Bulls,Chicago,Eastern,Central
5,CLE,Cleveland Cavaliers,Cleveland,Eastern,Central
6,DAL,Dallas Mavericks,Dallas,Western,Southwest
7,DEN,Denver Nuggets,Denver,Western,Northwest


### 8a. The naive join — and the two numbers to check immediately

After **any** join: (1) did the row count change? (2) how many right-side values are missing?

In [24]:
joined = games.merge(teams, left_on="opponent", right_on="abbr", how="left")

print("rows before:", len(games), "  rows after:", len(joined), "  ← unchanged, so no duplicate keys in teams")
print("games with no team match:", joined["team"].isna().sum())

show(joined[game_cols + ["abbr", "team", "conference", "division"]].sort_values("team", na_position="first"),
     "games ⟕ teams (left join) — unmatched games first", "purple",
     sources={**{c: "blue" for c in game_cols}, "opponent": "green", "abbr": "green",
              "team": "orange", "conference": "orange", "division": "orange"}, key=["opponent", "abbr"])

rows before: 1103   rows after: 1103   ← unchanged, so no duplicate keys in teams
games with no team match: 138


,season_label,date,location,opponent,outcome,points,abbr,team,conference,division
2,1996-97,1996-11-06,Away,CHH,L,5,nan,nan,nan,nan
25,1996-97,1997-01-05,Away,VAN,W,16,nan,nan,nan,nan
27,1996-97,1997-01-08,Home,CHH,W,9,nan,nan,nan,nan
29,1996-97,1997-01-14,Home,VAN,W,2,nan,nan,nan,nan
33,1996-97,1997-01-26,Away,SEA,W,7,nan,nan,nan,nan
36,1996-97,1997-02-02,Home,WSB,W,13,nan,nan,nan,nan
41,1996-97,1997-02-16,Home,SEA,L,0,nan,nan,nan,nan
43,1996-97,1997-02-21,Home,VAN,W,2,nan,nan,nan,nan


**What happened:** 138 games came back with pink cells. **Question: which opponents failed to match?** That is a left anti join on the key.

In [25]:
unmatched = games[~games["opponent"].isin(teams["abbr"])]["opponent"].value_counts().reset_index()
unmatched.columns = ["opponent", "games"]
show(unmatched, "opponent abbreviations with NO row in teams (left anti join)", "purple",
     sources={"opponent": "green", "games": "yellow"}, key="opponent")

never_faced = teams[~teams["abbr"].isin(games["opponent"])]
show(never_faced, "teams Kobe never faced under these keys (right anti join)", "purple",
     sources={"abbr": "green", "team": "orange", "city": "orange", "conference": "orange", "division": "orange"}, key="abbr")

,opponent,games
0,SEA,43
1,NJN,23
2,NOH,22
3,VAN,18
4,CHA,13
5,CHH,11
6,NOK,7
7,WSB,1


,abbr,team,city,conference,division
2,BRK,Brooklyn Nets,Brooklyn,Eastern,Atlantic
3,CHO,Charlotte Hornets,Charlotte,Eastern,Southeast
13,LAL,Los Angeles Lakers,Los Angeles,Western,Pacific
18,NOP,New Orleans Pelicans,New Orleans,Western,Southwest


**Interpretation:** every unmatched abbreviation is a franchise that moved or was renamed after Kobe played it (Seattle → Oklahoma City, New Jersey → Brooklyn, Vancouver → Memphis, the Hornets/Bobcats/Pelicans saga). The right anti join shows the *new* names of those same franchises — plus LAL, Kobe's own team, which he never played against. The keys are not wrong; they are from a different era than the lookup. **This is the everyday reality of joins: keys drift.**

### 8b. Fix the keys with a second join, then validate

**Question:** *How do we translate old abbreviations into current ones without touching games that already match?*
**Plan:** left-join the `moves` table (old → current) onto games; where a current abbreviation exists use it, otherwise keep the original. `fillna` does the "otherwise".

In [26]:
show(moves[["old_abbr", "old_name", "current_abbr"]], "moves — the translation table (key: old_abbr)", "orange", key="old_abbr", rows=8)

step = games.merge(moves[["old_abbr", "current_abbr"]], left_on="opponent", right_on="old_abbr", how="left")
step["opp_current"] = step["current_abbr"].fillna(step["opponent"])
show(step[["season_label", "opponent", "old_abbr", "current_abbr", "opp_current"]].drop_duplicates("opponent").sort_values("opponent"),
     "one row per opponent: how opp_current was built", "purple",
     sources={"season_label": "blue", "opponent": "green", "old_abbr": "green", "current_abbr": "orange", "opp_current": "yellow"},
     key=["opponent", "old_abbr"], rows=34)

games = step.drop(columns=["old_abbr", "current_abbr"])

,old_abbr,old_name,current_abbr
0,CHH,Charlotte Hornets,NOP
1,NOH,New Orleans Hornets,NOP
2,NOK,New Orleans/Oklahoma City Hornets,NOP
3,CHA,Charlotte Bobcats,CHO
4,NJN,New Jersey Nets,BRK
5,SEA,Seattle SuperSonics,OKC
6,VAN,Vancouver Grizzlies,MEM
7,WSB,Washington Bullets,WAS


,season_label,opponent,old_abbr,current_abbr,opp_current
4,1996-97,ATL,nan,nan,ATL
12,1996-97,BOS,nan,nan,BOS
607,2004-05,CHA,CHA,CHO,CHO
2,1996-97,CHH,CHH,NOP,NOP
18,1996-97,CHI,nan,nan,CHI
42,1996-97,CLE,nan,nan,CLE
31,1996-97,DAL,nan,nan,DAL
14,1996-97,DEN,nan,nan,DEN
13,1996-97,DET,nan,nan,DET
9,1996-97,GSW,nan,nan,GSW


In [27]:
joined = games.merge(teams, left_on="opp_current", right_on="abbr", how="left",
                     validate="many_to_one", indicator=True)
print(joined["_merge"].value_counts().to_string())

_merge
both          1103
left_only        0
right_only       0


**What happened:** `validate="many_to_one"` asserted that `abbr` is unique in `teams` (it is) and `indicator=True` proved every game is now `both`. The question we started with is now one `groupby` away:

In [28]:
by_division = (joined.groupby("division")
                     .agg(games=("points", "size"), avg_points=("points", "mean"),
                          win_share=("outcome", lambda s: (s == "W").mean()))
                     .round(3).sort_values("avg_points", ascending=False).reset_index())
show(by_division, "points and win share by opponent division — the answer", "purple",
     sources={"division": "green", "games": "yellow", "avg_points": "yellow", "win_share": "yellow"}, key="division")

,division,games,avg_points,win_share
0,Pacific,219,26.187000,0.735000
1,Northwest,260,25.408000,0.627000
2,Southwest,245,25.188000,0.653000
3,Atlantic,129,25.031000,0.690000
4,Southeast,115,25.009000,0.609000
5,Central,135,24.081000,0.667000


**Answer:** Kobe scored most against the Pacific division (his own, 26.2 per game, 73.5% wins) and least against the Central (24.1). The Southeast was his toughest by win share (60.9%). Neither number existed in either table alone — the join created the possibility of asking.

### 8c. The duplicate trap on real data

**Question:** *What if someone joins franchise history onto games by the current abbreviation?* `moves` has **three** rows for NOP (CHH, NOH, NOK). **Expect:** games against that franchise tripled.

In [29]:
history = games.merge(moves, left_on="opp_current", right_on="current_abbr", how="inner")
print("games vs. a relocated franchise:", games["opp_current"].isin(moves["current_abbr"]).sum())
print("rows after the join:            ", len(history))

nop = history[history["opp_current"] == "NOP"][["season_label", "date", "opponent", "opp_current", "old_abbr", "old_name"]]
show(nop, "one game vs. NOP became three rows", "purple",
     sources={"season_label": "blue", "date": "blue", "opponent": "blue", "opp_current": "green",
              "old_abbr": "orange", "old_name": "orange"}, key="opp_current", rows=6)

try:
    games.merge(moves, left_on="opp_current", right_on="current_abbr", how="inner", validate="many_to_one")
except pd.errors.MergeError:
    print('validate="many_to_one" raised MergeError — exactly what we want')

games vs. a relocated franchise: 209
rows after the join:             289


,season_label,date,opponent,opp_current,old_abbr,old_name
0,1996-97,1996-11-06,CHH,NOP,CHH,Charlotte Hornets
1,1996-97,1996-11-06,CHH,NOP,NOH,New Orleans Hornets
2,1996-97,1996-11-06,CHH,NOP,NOK,New Orleans/Oklahoma City Hornets
4,1996-97,1997-01-08,CHH,NOP,CHH,Charlotte Hornets
5,1996-97,1997-01-08,CHH,NOP,NOH,New Orleans Hornets
6,1996-97,1997-01-08,CHH,NOP,NOK,New Orleans/Oklahoma City Hornets


validate="many_to_one" raised MergeError — exactly what we want


### 8d. Suffixes and pivots on real data: home vs. away

**Question:** *Did Kobe score more at home than on the road, season by season?*
**Why a join (and then why not):** two grouped summaries — home points and away points per season — share the column name `points`, so joining them needs suffixes. Then notice the same table is really a *pivot* of one summary.

In [30]:
home = games[games["location"] == "Home"].groupby("season_label")["points"].mean().reset_index()
away = games[games["location"] == "Away"].groupby("season_label")["points"].mean().reset_index()
show(home, "home — mean points per season", "blue", key="season_label", rows=5)
show(away, "away — mean points per season", "orange", key="season_label", rows=5)

home_away = home.merge(away, on="season_label", suffixes=("_home", "_away"))
home_away["home_edge"] = (home_away["points_home"] - home_away["points_away"]).round(2)
show(home_away, 'home.merge(away, on="season_label", suffixes=("_home", "_away")) + home_edge', "purple",
     sources={"season_label": "green", "points_home": "blue", "points_away": "orange", "home_edge": "yellow"},
     key="season_label", rows=15)

,season_label,points
0,1996-97,8.656250
1,1997-98,16.512195
2,1998-99,19.960000
3,1999-00,22.696970
4,2000-01,28.500000


,season_label,points
0,1996-97,6.717949
1,1997-98,14.289474
2,1998-99,19.880000
3,1999-00,22.303030
4,2000-01,28.500000


,season_label,points_home,points_away,home_edge
0,1996-97,8.656250,6.717949,1.940000
1,1997-98,16.512195,14.289474,2.220000
2,1998-99,19.960000,19.880000,0.080000
3,1999-00,22.696970,22.303030,0.390000
4,2000-01,28.500000,28.500000,0.000000
5,2001-02,24.717949,25.731707,-1.010000
6,2002-03,29.756098,30.268293,-0.510000
7,2003-04,23.000000,25.066667,-2.070000
8,2004-05,28.187500,26.970588,1.220000
9,2005-06,36.975000,33.825000,3.150000


In [31]:
pt = games.pivot_table(index="season_label", columns="location", values="points", aggfunc="mean").round(1).reset_index()
pt.columns.name = None
show(pt, 'games.pivot_table(index="season_label", columns="location", values="points")  — same table, no join', "purple",
     sources={"season_label": "green", "Away": "yellow", "Home": "yellow"}, key="season_label", rows=15)

,season_label,Away,Home
0,1996-97,6.700000,8.700000
1,1997-98,14.300000,16.500000
2,1998-99,19.900000,20.000000
3,1999-00,22.300000,22.700000
4,2000-01,28.500000,28.500000
5,2001-02,25.700000,24.700000
6,2002-03,30.300000,29.800000
7,2003-04,25.100000,23.000000
8,2004-05,27.000000,28.200000
9,2005-06,33.800000,37.000000


---
## 9. Summary: the question decides the join

| The question sounds like… | You need | Because |
|---|---|---|
| "…for the ones that appear in both" | inner join | unmatched rows are noise |
| "…keep all of *these*, add info where it exists" | left join | the left table is the population; `NaN` = "no info" |
| "…which of these have no counterpart?" | left anti join | the *absence* of a match is the answer |
| "…audit both sources" | outer join + `indicator` | you need to see every row and where it came from |
| "…every combination of" | cross join | there is no key; you want all pairs |
| "same kind of rows, more of them" | `concat` (rows) | stacking, not matching |
| "side by side per period / category" | `pivot` / `pivot_table` | reshape one table instead of joining two |
| "my row count changed" | `validate=`, `drop_duplicates` | a duplicated key multiplied rows |

Watch any of these on the interactive page: [https://gato365.github.io/gsb5544_instructor_learn_prep/week_3/pandas.html](https://gato365.github.io/gsb5544_instructor_learn_prep/week_3/pandas.html).